# Allen CCF → 3D-printable STL

Pulls the Allen Mouse Brain Common Coordinate Framework structure mesh
via the AllenSDK (`ReferenceSpaceApi.download_structure_mesh`),
optionally cleans it with `trimesh`, rescales to a printable size,
exports STL, and visualizes the result with matplotlib.

Default target: whole-brain root (`structure_id=997`). Swap in
`315` for cortex, `688` for isocortex, `385` for V1, etc. Structure
IDs come from the Allen ontology.


In [ ]:
import os
from pathlib import Path
import numpy as np
import trimesh
from allensdk.api.queries.reference_space_api import ReferenceSpaceApi
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

REPO_ROOT = Path('../..').resolve()
OUT_DIR = REPO_ROOT / 'data' / 'ccf_stl'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('output dir:', OUT_DIR)

In [ ]:
# Pick which structure(s) to print. Each entry: (id, name, color).
# 997 = whole-brain root; 315 = cerebral cortex; 688 = isocortex.
STRUCTURES = [
    (997, 'root', '#d6b8a3'),
]
CCF_VERSION = ReferenceSpaceApi.CCF_2017
rsa = ReferenceSpaceApi()

meshes = {}
for sid, name, color in STRUCTURES:
    obj_path = OUT_DIR / f'{sid}_{name}.obj'
    if not obj_path.exists():
        print(f'downloading {name} (id={sid}) → {obj_path.name}')
        rsa.download_structure_mesh(structure_id=sid,
                                     ccf_version=CCF_VERSION,
                                     file_name=str(obj_path))
    else:
        print(f'cached {name} (id={sid}) → {obj_path.name}')
    m = trimesh.load(obj_path, force='mesh')
    meshes[sid] = (name, color, m)
    print(f'  {name}: vertices={len(m.vertices):,}  '
          f'faces={len(m.faces):,}  '
          f'watertight={m.is_watertight}')

In [ ]:
# Clean + rescale to a 3D-printable size.
# Allen meshes are in micrometers; scale so the longest axis = TARGET_MM
# millimetres. Default: 5x real-life mouse brain (real AP ≈ 13.2 mm
# → ≈ 66 mm print). Change SCALE_FACTOR to rescale globally.
SCALE_FACTOR = 5.0
_root_ext_max_um = meshes[997][2].bounding_box.extents.max()
TARGET_MM = SCALE_FACTOR * _root_ext_max_um / 1000.0
print(f'TARGET_MM = {TARGET_MM:.2f} mm  ({SCALE_FACTOR}x real life)')

printable = {}
for sid, (name, color, m) in meshes.items():
    mm = m.copy()
    # trimesh 4 cleanup: dedupe, drop degenerates, run repair.
    mm.merge_vertices()
    mm.update_faces(mm.unique_faces())
    mm.update_faces(mm.nondegenerate_faces())
    mm.remove_unreferenced_vertices()
    if not mm.is_watertight:
        try:
            trimesh.repair.fill_holes(mm)
            trimesh.repair.fix_normals(mm)
        except Exception as e:
            print(f'  {name}: repair failed: {e}')
    extents = mm.bounding_box.extents
    scale = TARGET_MM / extents.max()
    mm.apply_scale(scale)
    # Center on origin so the print starts from a clean reference.
    mm.apply_translation(-mm.bounding_box.centroid)
    printable[sid] = (name, color, mm)
    print(f'{name}: scale={scale:.6f}  '
          f'extents (mm)={mm.bounding_box.extents.round(2).tolist()}  '
          f'watertight={mm.is_watertight}  '
          f'volume={mm.volume:.2f} mm^3')

In [ ]:
# Save STL.
saved = []
for sid, (name, color, mm) in printable.items():
    stl_path = OUT_DIR / f'allen_ccf_{sid}_{name}_{int(TARGET_MM)}mm.stl'
    mm.export(stl_path)
    sz = stl_path.stat().st_size
    saved.append(stl_path)
    print(f'wrote {stl_path.name} ({sz/1024:.1f} kB, '
          f'{len(mm.faces):,} faces)')

In [ ]:
# Visualize. Decimate a viz copy for matplotlib (the on-disk STL keeps
# the full resolution), then shade each face by its normal dotted with
# a fixed light direction so the brain reads as a solid surface.
from matplotlib.colors import to_rgb

VIZ_FACE_TARGET = 30000
LIGHT_DIR = np.array([0.4, 0.4, 0.8])
LIGHT_DIR = LIGHT_DIR / np.linalg.norm(LIGHT_DIR)
AMBIENT = 0.35

def decimate_for_viz(m, target=VIZ_FACE_TARGET):
    if len(m.faces) <= target:
        return m
    try:
        return m.simplify_quadric_decimation(face_count=target)
    except Exception:
        step = int(np.ceil(len(m.faces) / target))
        keep = np.arange(0, len(m.faces), step)
        return trimesh.Trimesh(vertices=m.vertices,
                                faces=m.faces[keep], process=False)

def shade_facecolors(m, base_rgb):
    n = m.face_normals
    intensity = np.clip(n @ LIGHT_DIR, 0, 1) * (1 - AMBIENT) + AMBIENT
    base = np.array(to_rgb(base_rgb))
    rgb = intensity[:, None] * base[None, :]
    return np.clip(rgb, 0, 1)

viz = {sid: (n, c, decimate_for_viz(mm))
        for sid, (n, c, mm) in printable.items()}

fig = plt.figure(figsize=(14, 5.5), dpi=140, facecolor='white')
views = [('iso',     (25, -60)),
         ('lateral', (0, 0)),
         ('dorsal',  (90, -90))]
all_pts = np.vstack([mv.vertices for _, _, mv in viz.values()])
mn, mx = all_pts.min(axis=0), all_pts.max(axis=0)
ctr = (mn + mx) / 2
rad = (mx - mn).max() / 2

for i, (vname, (elev, azim)) in enumerate(views):
    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    ax.set_facecolor('white')
    for sid, (name, color, mv) in viz.items():
        face_rgb = shade_facecolors(mv, color)
        poly = Poly3DCollection(mv.triangles,
                                  facecolors=face_rgb,
                                  edgecolors='none',
                                  linewidths=0.0,
                                  alpha=1.0)
        poly.set_zsort('average')
        ax.add_collection3d(poly)
    ax.set_xlim(ctr[0] - rad, ctr[0] + rad)
    ax.set_ylim(ctr[1] - rad, ctr[1] + rad)
    ax.set_zlim(ctr[2] - rad, ctr[2] + rad)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(vname, fontsize=11)
    ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)'); ax.set_zlabel('Z (mm)')
    ax.tick_params(labelsize=7)
fig.suptitle(f'Allen CCF (root, scaled to {int(TARGET_MM)} mm) · '
              f'{sum(len(mm.faces) for _, _, mm in printable.values()):,} '
              f'faces saved · {sum(len(mv.faces) for _, _, mv in viz.values()):,} '
              f'shown',
              fontsize=12)
plt.tight_layout()
fig.savefig(OUT_DIR / 'ccf_preview.png', dpi=140, bbox_inches='tight',
             facecolor='white')
plt.show()
print('preview saved to', OUT_DIR / 'ccf_preview.png')

In [ ]:
# Cleaner 3D preview using pyvista (VTK). Renders off-screen and writes
# a screenshot. Matplotlib's z-sorting struggles with closed meshes; VTK
# does proper depth-buffered rendering.
import pyvista as pv

pv.OFF_SCREEN = True
pv.set_plot_theme('document')

views = [('iso',     [(1.2,  1.2, 1.0), (0, 0, 0), (0, 0, 1)]),
         ('lateral', [(0,    0,   2.4), (0, 0, 0), (0, 1, 0)]),
         ('dorsal',  [(0,    2.4, 0),   (0, 0, 0), (1, 0, 0)])]

pl = pv.Plotter(shape=(1, len(views)),
                 window_size=(1500, 600),
                 off_screen=True)
for j, (vname, cam) in enumerate(views):
    pl.subplot(0, j)
    for sid, (name, color, mm) in printable.items():
        pv_mesh = pv.PolyData(mm.vertices,
                                np.hstack([np.full((len(mm.faces), 1), 3),
                                             mm.faces]).astype(np.int64))
        pl.add_mesh(pv_mesh, color=color, smooth_shading=True,
                     specular=0.25, specular_power=20, ambient=0.25,
                     diffuse=0.9)
    pl.add_text(vname, font_size=12)
    pl.camera_position = cam
    pl.reset_camera()
pl.background_color = 'white'
pv_png = OUT_DIR / 'ccf_preview_pv.png'
pl.screenshot(str(pv_png))
pl.close()
print('pyvista preview →', pv_png)

# Show inline.
from IPython.display import Image, display
display(Image(filename=str(pv_png)))

---

## Assembly design — cortex hat with etched areal boundaries

Plan: 3-piece print
1. **Cortex hat** — Isocortex (688) with engraved boundaries between
   the ~40 cortical areas (children of Isocortex in the Allen ontology).
2. **Core block** — subcortex + thalamus + hippocampus + brainstem +
   cerebellum, fused into one piece with engraved major-division
   boundaries on its outer surface.
3. Magnet pockets at the cortex/core mating surface.

Boundary etching procedure:
1. Sample the Allen 25 µm annotation volume at each vertex of the
   structure mesh.
2. Roll up each leaf label to its cortical-area ancestor (or, for
   the core, to a chosen "major division" set).
3. A vertex is on a boundary if any incident face has a neighboring
   vertex with a different label.
4. Displace boundary vertices inward along the vertex normal by
   `ETCH_DEPTH_MM` (default 0.8 mm in the scaled print).

In [ ]:
# Download annotation volume (25 µm) + structure tree.
from allensdk.core.reference_space_cache import ReferenceSpaceCache
RES_UM = 25
manifest = OUT_DIR / 'reference_space_manifest.json'
rspc = ReferenceSpaceCache(resolution=RES_UM,
                            reference_space_key='annotation/ccf_2017',
                            manifest=str(manifest))
ann, ann_meta = rspc.get_annotation_volume()
tree = rspc.get_structure_tree()
print(f'annotation volume: {ann.shape}  dtype={ann.dtype}  '
      f'voxel={RES_UM} µm')

# Cortical-area set: descendants of Isocortex whose children are all
# layer-specific subdivisions (acronym ends in a digit, "2/3", "6a", or
# "6b"). This is the right "area" level — yields ~40 entries like VISp,
# MOp, SSp-bfd, ACAd, etc.
ISO_ID = 688
def is_layer_acro(acro):
    return (acro.endswith('2/3') or acro.endswith('6a')
             or acro.endswith('6b')
             or (len(acro) >= 2 and acro[-1].isdigit()))

def child_ids_of(sid):
    return tree.child_ids([int(sid)])[0]

iso_desc = tree.descendant_ids([ISO_ID])[0]
iso_kid_set = set()
for sid in iso_desc:
    if sid == ISO_ID:
        continue
    kids = child_ids_of(sid)
    if not kids:
        continue
    kid_acros = [tree.get_structures_by_id([int(k)])[0]['acronym']
                  for k in kids]
    if all(is_layer_acro(a) for a in kid_acros):
        iso_kid_set.add(int(sid))
print(f'cortical areas detected: {len(iso_kid_set)}')
for sid in sorted(iso_kid_set):
    s = tree.get_structures_by_id([sid])[0]
    print(f'  {sid:6d}  {s["acronym"]:<10s}  {s["name"]}')

In [ ]:
# Download isocortex mesh, label every vertex by cortical-area ancestor,
# then SMOOTH the labels via diffusion of indicator fields on the mesh.
iso_obj_path = OUT_DIR / f'{ISO_ID}_isocortex.obj'
if not iso_obj_path.exists():
    print(f'downloading isocortex (id={ISO_ID})')
    rsa.download_structure_mesh(structure_id=ISO_ID,
                                  ccf_version=CCF_VERSION,
                                  file_name=str(iso_obj_path))
cortex = trimesh.load(iso_obj_path, force='mesh')
cortex.merge_vertices()
cortex.update_faces(cortex.unique_faces())
cortex.update_faces(cortex.nondegenerate_faces())
cortex.remove_unreferenced_vertices()

# Subdivide once so the boundary curves have sub-mesh-vertex precision.
SUBDIVIDE = True
if SUBDIVIDE:
    nv, nf = trimesh.remesh.subdivide(cortex.vertices, cortex.faces)
    cortex = trimesh.Trimesh(vertices=nv, faces=nf, process=False)
print(f'cortex mesh: vertices={len(cortex.vertices):,}  '
      f'faces={len(cortex.faces):,}')

# Sample the annotation INWARD of each vertex (pia is outside the
# labeled volume). 400 µm in is well inside the cortical ribbon.
INWARD_STEP_UM = 400.0
verts_um = cortex.vertices.copy()
verts_inward = verts_um - INWARD_STEP_UM * cortex.vertex_normals
vox = np.round(verts_inward / RES_UM).astype(int)
vox = np.clip(vox, 0, np.array(ann.shape) - 1)
leaf_labels = ann[vox[:, 0], vox[:, 1], vox[:, 2]].astype(np.int64)

# Roll up each leaf annotation label to its cortical-area ancestor.
leaf_to_area = {0: 0}
for leaf in np.unique(leaf_labels):
    if leaf == 0:
        continue
    try:
        path = tree.get_structures_by_id([int(leaf)])[0]['structure_id_path']
    except Exception:
        path = []
    match = [s for s in (path or []) if int(s) in iso_kid_set]
    leaf_to_area[int(leaf)] = int(match[0]) if match else 0
vert_area = np.array([leaf_to_area[int(l)] for l in leaf_labels],
                       dtype=np.int64)
n_unlabeled = int((vert_area == 0).sum())
print(f'unlabeled cortex vertices: {n_unlabeled}/{len(vert_area)} '
      f'({100*n_unlabeled/len(vert_area):.1f}%)')

# Fill unlabeled vertices from nearest labeled neighbor.
from scipy.spatial import cKDTree
labeled_mask = vert_area != 0
if (~labeled_mask).any() and labeled_mask.any():
    kd = cKDTree(verts_um[labeled_mask])
    _, ii = kd.query(verts_um[~labeled_mask])
    vert_area[~labeled_mask] = vert_area[np.where(labeled_mask)[0][ii]]

# Build normalized mesh adjacency W (row-stochastic).
from scipy.sparse import csr_matrix, diags
N = len(cortex.vertices)
edges = cortex.edges_unique
rows = np.concatenate([edges[:, 0], edges[:, 1]])
cols = np.concatenate([edges[:, 1], edges[:, 0]])
adj = csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                  shape=(N, N))
deg = np.asarray(adj.sum(axis=1)).ravel()
W = diags(1.0 / np.maximum(deg, 1.0)).dot(adj).tocsr()

# DIFFUSE label indicator fields. For each area A we initialize a
# scalar field = 1 where vert_area == A else 0, then run K rounds of
# averaging with W. After many rounds, the indicator fields are smooth
# scalar fields, and argmax across A gives clean Cheeger-like
# boundary curves rather than the jagged edges 1-ring majority voting
# gets stuck on.
unique_areas = np.array(sorted(set(int(a) for a in vert_area if a > 0)))
S = np.zeros((N, len(unique_areas)), dtype=np.float32)
for j, A in enumerate(unique_areas):
    S[:, j] = (vert_area == int(A)).astype(np.float32)

DIFFUSE_ITERS = 200
ALPHA = 0.5
for _ in range(DIFFUSE_ITERS):
    S = (1 - ALPHA) * S + ALPHA * (W @ S)
vert_area_new = unique_areas[np.argmax(S, axis=1)]
n_changed = int((vert_area_new != vert_area).sum())
vert_area = vert_area_new
print(f'after {DIFFUSE_ITERS}-iter diffusion: {n_changed} labels changed, '
      f'{len(set(vert_area.tolist()))} unique areas')

In [ ]:
# Boundary detection + curve smoothing + smooth groove etch + STL.
ROOT_EXTENT_UM = meshes[997][2].bounding_box.extents.max()
SHARED_SCALE = TARGET_MM / ROOT_EXTENT_UM
print(f'shared scale: {SHARED_SCALE:.6f}')

# Boundary FACES: faces whose three vertices don't all share a label.
fv = vert_area[cortex.faces]
boundary_face = (fv[:, 0] != fv[:, 1]) | \
                 (fv[:, 0] != fv[:, 2]) | \
                 (fv[:, 1] != fv[:, 2])
boundary_vert_idx = np.unique(cortex.faces[boundary_face].ravel())
print(f'boundary faces: {boundary_face.sum():,}/{len(cortex.faces):,}  '
      f'boundary verts: {len(boundary_vert_idx):,}')

# Smooth the POSITIONS of boundary vertices along the boundary-only
# subgraph. The mesh itself stays where it is — we use these smoothed
# phantom positions as the reference curve for the etch distance field.
# Effect: the groove follows a clean Laplacian-smoothed curve rather
# than the original jagged boundary vertex set.
b_adj = adj[boundary_vert_idx][:, boundary_vert_idx]
b_inv_deg = 1.0 / np.maximum(np.asarray(b_adj.sum(axis=1)).ravel(), 1.0)
b_W = diags(b_inv_deg).dot(b_adj).tocsr()

b_pos = cortex.vertices[boundary_vert_idx].copy().astype(np.float32)
CURVE_SMOOTH_ITERS = 80
CURVE_ALPHA = 0.5
for _ in range(CURVE_SMOOTH_ITERS):
    b_pos = (1 - CURVE_ALPHA) * b_pos + CURVE_ALPHA * (b_W @ b_pos)

# Apply shared scale (centered so all pieces align).
cortex_print = cortex.copy()
cortex_print.apply_scale(SHARED_SCALE)
root_ctr = meshes[997][2].bounding_box.centroid * SHARED_SCALE
cortex_print.apply_translation(-root_ctr)
b_pos_print = b_pos * SHARED_SCALE - root_ctr

# Half-cosine depth profile: depth = ETCH at distance 0, decays smoothly
# to 0 at GROOVE_HALF_WIDTH_MM. This gives clean U-shaped grooves with
# continuous derivatives.
ETCH_DEPTH_MM        = 1.0
GROOVE_HALF_WIDTH_MM = 1.0

kd_bd = cKDTree(b_pos_print)
dist, _ = kd_bd.query(cortex_print.vertices, k=1)
profile = np.cos(np.pi/2 * np.clip(dist / GROOVE_HALF_WIDTH_MM, 0, 1)) ** 2

vn = cortex_print.vertex_normals.copy()
cortex_print.vertices = cortex_print.vertices - vn * (ETCH_DEPTH_MM * profile)[:, None]

# Final polish: light Taubin smoothing (non-shrinking) on the full mesh.
# Cleans up residual high-frequency wiggle in the grooves without
# closing them up.
trimesh.smoothing.filter_taubin(cortex_print, iterations=10)

print(f'cortex piece: extents (mm)='
      f'{cortex_print.bounding_box.extents.round(2).tolist()}  '
      f'faces={len(cortex_print.faces):,}')

cortex_stl = OUT_DIR / f'allen_ccf_cortex_hat_etched_{int(TARGET_MM)}mm.stl'
cortex_print.export(cortex_stl)
print(f'wrote {cortex_stl.name} '
      f'({cortex_stl.stat().st_size/1024:.1f} kB)')

# Per-face dominant area (majority vote; defaults to v0).
face_area = fv[:, 0]
m12 = fv[:, 1] == fv[:, 2]
m01 = fv[:, 0] == fv[:, 1]
face_area = np.where(m12 & ~m01, fv[:, 1], face_area)

In [ ]:
# Preview: 4-view render with correct CCF axes.
# CCF axes: X=AP, Y=DV (+ventral), Z=ML. So "lateral" = look along Z.
import pyvista as pv
pv.OFF_SCREEN = True
pv.set_plot_theme('document')

areas_present = sorted({int(a) for a in face_area if a > 0})
import matplotlib as mpl
pal = (list(mpl.colormaps['tab20'].colors) +
       list(mpl.colormaps['tab20b'].colors) +
       list(mpl.colormaps['tab20c'].colors))
area_color = {sid: pal[i % len(pal)] for i, sid in enumerate(areas_present)}
face_rgb = np.array([area_color.get(int(a), (0.6, 0.6, 0.6))
                      for a in face_area])
face_rgb_solid = np.full_like(face_rgb, fill_value=0.83)

bb = cortex_print.bounding_box.extents
d = bb.max() * 2.2
views = [
    ('dorsal',  [(0, -d, 0), (0, 0, 0), (-1, 0,  0)]),
    ('right',   [(0,  0,  d), (0, 0, 0), ( 0, -1, 0)]),
    ('caudal',  [(d,  0,  0), (0, 0, 0), ( 0, -1, 0)]),
    ('oblique', [(0.7*d, -0.5*d, 0.7*d), (0, 0, 0), (0, -1, 0)]),
]

def render(face_colors, fname, title):
    pl = pv.Plotter(shape=(2, 2), window_size=(1600, 1400),
                     off_screen=True)
    for k, (vname, cam) in enumerate(views):
        pl.subplot(k // 2, k % 2)
        poly = pv.PolyData(
            cortex_print.vertices,
            np.hstack([np.full((len(cortex_print.faces), 1), 3),
                         cortex_print.faces]).astype(np.int64))
        poly.cell_data['rgb'] = (face_colors * 255).astype(np.uint8)
        pl.add_mesh(poly, scalars='rgb', rgb=True,
                     smooth_shading=False, specular=0.20,
                     ambient=0.20, diffuse=0.95, show_edges=False)
        pl.add_text(vname, font_size=10)
        pl.camera_position = cam
        pl.reset_camera()
    pl.background_color = 'white'
    pl.screenshot(str(fname))
    pl.close()
    print(f'wrote {fname.name}')

render(face_rgb,
        OUT_DIR / 'cortex_hat_etched_color.png',
        'colored by area')
render(face_rgb_solid,
        OUT_DIR / 'cortex_hat_etched_solid.png',
        'solid (grooves only)')

from IPython.display import Image, display
print('Colored by area:')
display(Image(filename=str(OUT_DIR / 'cortex_hat_etched_color.png')))
print('Solid — etched grooves visible:')
display(Image(filename=str(OUT_DIR / 'cortex_hat_etched_solid.png')))

### Labeled variant — etched cortical-area acronyms (VISp, MOp, …)

For each cortical area with enough dorsal-facing surface, etch the
Allen acronym (Helvetica Bold) into the cortex at the area's dorsal
centroid. Key step: project every cortex vertex into the label's local
(u, v) tangent frame and read the bitmap there — handles surface
curvature correctly (a flat-plane KD-tree-of-ink approach loses too
many vertices on the curved cortex).

To get crisp letter strokes we subdivide the cortex once more
(208k → 835k verts) and rebuild the label and groove fields on the
finer mesh. Area grooves are kept but shallower so the labels dominate.

In [ ]:
# Self-contained labeled-cortex pipeline. Reuses earlier objects:
#   ann, tree, iso_kid_set, ISO_ID, RES_UM, SHARED_SCALE, root_ctr,
#   leaf_to_area, TARGET_MM.
# Produces: allen_ccf_cortex_hat_labeled_{TARGET_MM}mm.stl and
# preview PNGs cortex_hat_labeled_{solid,color}.png.
from PIL import Image, ImageDraw, ImageFont

# Fresh cortex at 2x subdivide (208k → 835k verts → ~0.15mm spacing).
cortex_lbl = trimesh.load(iso_obj_path, force='mesh')
cortex_lbl.merge_vertices()
cortex_lbl.update_faces(cortex_lbl.unique_faces())
cortex_lbl.update_faces(cortex_lbl.nondegenerate_faces())
cortex_lbl.remove_unreferenced_vertices()
for _ in range(2):
    nv2, nf2 = trimesh.remesh.subdivide(cortex_lbl.vertices,
                                          cortex_lbl.faces)
    cortex_lbl = trimesh.Trimesh(vertices=nv2, faces=nf2, process=False)
print(f'cortex (labeled prep): {len(cortex_lbl.vertices):,} verts, '
      f'{len(cortex_lbl.faces):,} faces')

# Re-label new vertices via annotation sampling (same trick as before).
v2 = cortex_lbl.vertices.copy()
v2_in = v2 - INWARD_STEP_UM * cortex_lbl.vertex_normals
vox2 = np.clip(np.round(v2_in / RES_UM).astype(int), 0,
                np.array(ann.shape) - 1)
leaf2 = ann[vox2[:, 0], vox2[:, 1], vox2[:, 2]].astype(np.int64)
va2 = np.array([leaf_to_area.get(int(l), 0) for l in leaf2], dtype=np.int64)
mask2 = va2 != 0
if (~mask2).any() and mask2.any():
    kd2 = cKDTree(v2[mask2])
    _, ii2 = kd2.query(v2[~mask2])
    va2[~mask2] = va2[np.where(mask2)[0][ii2]]

# Diffuse on the finer mesh.
N2 = len(cortex_lbl.vertices)
edges2 = cortex_lbl.edges_unique
r2 = np.concatenate([edges2[:, 0], edges2[:, 1]])
c2 = np.concatenate([edges2[:, 1], edges2[:, 0]])
adj2 = csr_matrix((np.ones(len(r2), dtype=np.float32), (r2, c2)),
                   shape=(N2, N2))
deg2 = np.asarray(adj2.sum(axis=1)).ravel()
W2 = diags(1.0 / np.maximum(deg2, 1.0)).dot(adj2).tocsr()
ua2 = np.array(sorted(set(int(a) for a in va2 if a > 0)))
S2 = np.zeros((N2, len(ua2)), dtype=np.float32)
for j, A in enumerate(ua2):
    S2[:, j] = (va2 == int(A)).astype(np.float32)
for _ in range(200):
    S2 = 0.5 * S2 + 0.5 * (W2 @ S2)
va2 = ua2[np.argmax(S2, axis=1)]

# Shallow area grooves + smoothed boundary curve (same recipe as before).
fv2 = va2[cortex_lbl.faces]
bf2 = (fv2[:, 0] != fv2[:, 1]) | (fv2[:, 0] != fv2[:, 2]) | \
      (fv2[:, 1] != fv2[:, 2])
bv2 = np.unique(cortex_lbl.faces[bf2].ravel())
b_adj2 = adj2[bv2][:, bv2]
b_W2 = diags(1.0 / np.maximum(np.asarray(b_adj2.sum(axis=1)).ravel(),
                                1.0)).dot(b_adj2).tocsr()
b_pos2 = cortex_lbl.vertices[bv2].copy().astype(np.float32)
for _ in range(80):
    b_pos2 = 0.5 * b_pos2 + 0.5 * (b_W2 @ b_pos2)

cl = cortex_lbl.copy()
cl.apply_scale(SHARED_SCALE)
cl.apply_translation(-root_ctr)
b_pos2_print = b_pos2 * SHARED_SCALE - root_ctr

# Shallow grooves so labels dominate (vs ETCH_DEPTH_MM = 1.0 earlier).
ETCH_DEPTH_LBL = 0.6
HALFW_LBL      = 0.7
kd_bd2 = cKDTree(b_pos2_print)
dd, _ = kd_bd2.query(cl.vertices, k=1)
prof2 = np.cos(np.pi/2 * np.clip(dd / HALFW_LBL, 0, 1)) ** 2
vn_init = cl.vertex_normals.copy()
cl.vertices = cl.vertices - vn_init * (ETCH_DEPTH_LBL * prof2)[:, None]
trimesh.smoothing.filter_taubin(cl, iterations=10)
print(f'labeled cortex base: {len(cl.faces):,} faces')

In [ ]:
# Etch cortical-area acronyms onto the labeled cortex.
# - Bilateral: separate label per hemisphere (split at z = 0).
# - PCA-rotated: text reading direction follows the area's principal axis.
# - Per-area sizing: font scales to fit each hemisphere's local extent.
# - Contained: only this area's vertices etch (`area_mask_full`).
# - Anti-aliased sharp edges: bilinear sampling of an AA bitmap.
# - Curated dorsal set: skip areas that don't read well from above.
area_acro = {int(sid): tree.get_structures_by_id([int(sid)])[0]['acronym']
              for sid in iso_kid_set}
DORSAL_AREAS = {
    'FRP', 'MOp', 'MOs',
    'SSp-ll', 'SSp-m', 'SSp-n', 'SSp-tr', 'SSp-ul', 'SSp-un',
    'SSs',
    'VISp', 'VISal', 'VISam', 'VISpm', 'VISpl', 'VISl', 'VISrl', 'VISa',
    'AUDp',
    'ACAd', 'ACAv',
    'RSPd', 'RSPv',
    'ORBm',
}
dorsal_ids = {sid for sid, a in area_acro.items() if a in DORSAL_AREAS}

FONT_PATH      = '/System/Library/Fonts/Helvetica.ttc'
FONT_INDEX     = 1                # 1 = Helvetica Bold within the TTC
FONT_PT        = 96
PAD_PX         = 4
LABEL_DEPTH_MM = 1.2
DEPTH_TOL_MM   = 3.0
DORSAL_NORM_TH = -0.55
FIT_FRAC_W     = 0.55             # label W <= 55% of area extent along PC1
FIT_FRAC_H     = 0.50             # label H <= 50% of area extent along PC2
MIN_LETTER_MM  = 1.6
MAX_LETTER_MM  = 3.5

def text_bitmap(text):
    """Anti-aliased text bitmap → float [0, 1]. Bilinear sampling of
    this field gives crisp letter edges on a polygonal mesh."""
    try:
        font = ImageFont.truetype(FONT_PATH, FONT_PT, index=FONT_INDEX)
    except OSError:
        font = ImageFont.truetype(FONT_PATH, FONT_PT)
    img0 = Image.new('L', (1, 1))
    bbox = ImageDraw.Draw(img0).textbbox((0, 0), text, font=font)
    w = bbox[2] - bbox[0] + 2 * PAD_PX
    h = bbox[3] - bbox[1] + 2 * PAD_PX
    img = Image.new('L', (w, h), color=255)
    ImageDraw.Draw(img).text(
        (PAD_PX - bbox[0], PAD_PX - bbox[1]), text, font=font, fill=0)
    return 1.0 - np.asarray(img, dtype=np.float32) / 255.0

def bilinear(field, col, row):
    H_, W_ = field.shape
    c0 = np.floor(col).astype(int); r0 = np.floor(row).astype(int)
    c1 = c0 + 1; r1 = r0 + 1
    cf = col - c0; rf = row - r0
    c0 = np.clip(c0, 0, W_ - 1); c1 = np.clip(c1, 0, W_ - 1)
    r0 = np.clip(r0, 0, H_ - 1); r1 = np.clip(r1, 0, H_ - 1)
    return (field[r0, c0] * (1 - cf) * (1 - rf) +
            field[r0, c1] * cf       * (1 - rf) +
            field[r1, c0] * (1 - cf) * rf       +
            field[r1, c1] * cf       * rf)

WORLD_RIGHT = np.array([0.0, 0.0, 1.0])
WORLD_DOWN  = np.array([1.0, 0.0, 0.0])
def tangent_frame(normal):
    u = WORLD_RIGHT - normal * (WORLD_RIGHT @ normal)
    if np.linalg.norm(u) < 1e-3:
        u = np.cross(normal, WORLD_DOWN)
    u = u / np.linalg.norm(u)
    v = WORLD_DOWN - normal * (WORLD_DOWN @ normal) - u * (WORLD_DOWN @ u)
    if np.linalg.norm(v) < 1e-3:
        v = np.cross(normal, u)
    v = v / np.linalg.norm(v)
    return u, v

vn_lbl = cl.vertex_normals.copy()
text_inkness = np.zeros(len(cl.vertices), dtype=np.float32)
placed = []
for area_id in sorted(iso_kid_set):
    if int(area_id) not in dorsal_ids:
        continue
    text = area_acro.get(int(area_id), '')
    if not text: continue
    bitmap = text_bitmap(text)
    H_px, W_px = bitmap.shape
    area_mask_full = va2 == int(area_id)
    if area_mask_full.sum() < 50: continue

    for hemi_name, hemi_sign in (('L', -1), ('R', +1)):
        hemi_mask = (area_mask_full
                      & (cl.vertices[:, 2] * hemi_sign > 0)
                      & (vn_lbl[:, 1] < DORSAL_NORM_TH))
        if hemi_mask.sum() < 80: continue
        hpts = cl.vertices[hemi_mask]; hnorms = vn_lbl[hemi_mask]

        # PCA on (X, Z) — text reading direction follows the area\'s
        # principal axis (so elongated areas get rotated labels).
        xz = hpts[:, [0, 2]]; xz_ctr = xz.mean(axis=0); xz_c = xz - xz_ctr
        cov = np.cov(xz_c.T) if len(xz_c) > 1 else np.eye(2)
        evals, evecs = np.linalg.eigh(cov)
        pc1 = evecs[:, 1].astype(float)
        proj1 = xz_c @ pc1; proj2 = xz_c @ evecs[:, 0]
        ext1 = float(proj1.max() - proj1.min())
        ext2 = float(proj2.max() - proj2.min())

        ps_for_w = (FIT_FRAC_W * ext1) / W_px
        ps_for_h = (FIT_FRAC_H * ext2) / H_px
        pixel_size_mm = min(ps_for_w, ps_for_h, MAX_LETTER_MM / H_px)
        letter_mm = pixel_size_mm * H_px
        if letter_mm < MIN_LETTER_MM: continue

        d_xz = np.linalg.norm(xz_c, axis=1)
        closest = np.argsort(d_xz)[:min(80, len(hpts))]
        best = closest[np.argmin(hpts[closest, 1])]
        site = hpts[best]; site_n = hnorms[best]

        pc1_3d = np.array([pc1[0], 0.0, pc1[1]])
        u_ax = pc1_3d - site_n * (pc1_3d @ site_n)
        nu = np.linalg.norm(u_ax)
        if nu < 1e-6:
            u_ax, _ = tangent_frame(site_n)
        else:
            u_ax = u_ax / nu
        v_ax = np.cross(site_n, u_ax)
        v_ax = v_ax / np.linalg.norm(v_ax)
        if v_ax[0] < 0:                # right-side-up in dorsal view
            v_ax = -v_ax; u_ax = -u_ax

        rel = cl.vertices - site
        u_co = rel @ u_ax; v_co = rel @ v_ax; w_co = rel @ site_n
        col = u_co / pixel_size_mm + W_px / 2
        row = v_co / pixel_size_mm + H_px / 2
        in_box = ((col >= 0) & (col < W_px - 1) &
                   (row >= 0) & (row < H_px - 1) &
                   (np.abs(w_co) < DEPTH_TOL_MM) &
                   area_mask_full)
        if not in_box.any(): continue
        ink_vals = bilinear(bitmap, col[in_box], row[in_box])
        idx = np.where(in_box)[0]
        text_inkness[idx] = np.maximum(text_inkness[idx], ink_vals)
        placed.append((text, hemi_name, letter_mm))

print(f'placed {len(placed)} labels')
print(f'inked vertices: {int((text_inkness > 0).sum())}')
cl.vertices = cl.vertices - vn_lbl * (LABEL_DEPTH_MM * text_inkness)[:, None]

cortex_lbl_stl = OUT_DIR / f'allen_ccf_cortex_hat_labeled_{int(TARGET_MM)}mm.stl'
cl.export(cortex_lbl_stl)
print(f'wrote {cortex_lbl_stl.name} '
      f'({cortex_lbl_stl.stat().st_size/1024/1024:.1f} MB, '
      f'{len(cl.faces):,} faces)')


In [ ]:
# Preview the labeled cortex from dorsal + zoom + oblique angles.
areas_present_lbl = sorted({int(a) for a in va2 if a > 0})
pal2 = (list(mpl.colormaps['tab20'].colors) +
        list(mpl.colormaps['tab20b'].colors) +
        list(mpl.colormaps['tab20c'].colors))
area_color_lbl = {sid: pal2[i % len(pal2)]
                   for i, sid in enumerate(areas_present_lbl)}
face_area_lbl = fv2[:, 0]
m12_l = fv2[:, 1] == fv2[:, 2]
m01_l = fv2[:, 0] == fv2[:, 1]
face_area_lbl = np.where(m12_l & ~m01_l, fv2[:, 1], face_area_lbl)
face_rgb_lbl   = np.array([area_color_lbl.get(int(a), (0.6,)*3)
                            for a in face_area_lbl])
face_rgb_solid = np.full_like(face_rgb_lbl, 0.85)

d = cl.bounding_box.extents.max() * 1.8
views_lbl = [
    ('dorsal',                    (0, -d, 0), (0, 0, 0), (-1, 0, 0), 1.0),
    ('dorsal (zoom, posterior)',  (0, -d, 0), (20, 0, 0), (-1, 0, 0), 4.0),
    ('dorsal (zoom, anterior)',   (0, -d, 0), (-30, 0, 0), (-1, 0, 0), 4.0),
    ('dorsal (zoom, mid)',        (0, -d, 0), (0, 0, 0), (-1, 0, 0), 4.0),
]

def render_lbl(face_colors, fname):
    pl = pv.Plotter(shape=(2, 2), window_size=(2200, 1800),
                     off_screen=True)
    for k, (vname, cam_pos, focal, up, zoom) in enumerate(views_lbl):
        pl.subplot(k // 2, k % 2)
        poly = pv.PolyData(
            cl.vertices,
            np.hstack([np.full((len(cl.faces), 1), 3),
                         cl.faces]).astype(np.int64))
        poly.cell_data['rgb'] = (face_colors * 255).astype(np.uint8)
        pl.add_mesh(poly, scalars='rgb', rgb=True,
                     smooth_shading=True, specular=0.15,
                     ambient=0.20, diffuse=0.95)
        pl.add_text(vname, font_size=10)
        pl.camera_position = (cam_pos, focal, up)
        pl.reset_camera()
        pl.camera.zoom(zoom)
    pl.background_color = 'white'
    pl.screenshot(str(fname))
    pl.close()
    print(f'wrote {fname.name}')

render_lbl(face_rgb_solid,
            OUT_DIR / 'cortex_hat_labeled_solid.png')
render_lbl(face_rgb_lbl,
            OUT_DIR / 'cortex_hat_labeled_color.png')

print('\nLabeled cortex — solid:')
display(Image(filename=str(OUT_DIR / 'cortex_hat_labeled_solid.png')))
print('\nLabeled cortex — colored:')
display(Image(filename=str(OUT_DIR / 'cortex_hat_labeled_color.png')))

---

## Inner-brain "core" — exploring two depictions

The core piece = everything inside the cortex hat (cerebral nuclei,
hippocampal formation, thalamus, hypothalamus, midbrain, hindbrain,
cerebellum, olfactory areas).

I build it from the Allen annotation volume directly: voxels whose
label is NOT a descendant of Isocortex are kept; marching cubes
extracts a closed surface.

Then I show two options:
- **A — surface engraving** (same trick as the cortex hat: boundaries
  between major divisions etched as grooves on the outer surface).
- **B — sagittal mid-line cut**, exposing internal regions on the
  flat cut face. Each pixel of the cut face is colored by which
  region sits at that voxel — far more informative for inner
  structure than surface engraving, but adds one piece (you can
  print one cut half, or both halves as a 2-piece core).

In [ ]:
# Major divisions we want to show on the inner-brain core.
MAJOR_DIVISIONS = {
    623:  ('CNU',  'Cerebral nuclei'),
    1089: ('HPF',  'Hippocampal formation'),
    703:  ('CTXsp','Cortical subplate'),
    698:  ('OLF',  'Olfactory areas'),
    549:  ('TH',   'Thalamus'),
    1097: ('HY',   'Hypothalamus'),
    313:  ('MB',   'Midbrain'),
    1065: ('HB',   'Hindbrain'),
    512:  ('CB',   'Cerebellum'),
}
MAJDIV_IDS = set(MAJOR_DIVISIONS.keys())

# Build a clean, topologically-closed core mask:
#   core = (brain envelope) ∩ (not isocortex).
# binary_fill_holes on (ann > 0) treats ventricles/tracts that
# happen to have annotation 0 as still inside the brain, so the
# resulting mesh has no internal holes.
from skimage import measure
from scipy.ndimage import gaussian_filter, binary_fill_holes
ctx_desc = list(tree.descendant_ids([ISO_ID])[0])
ctx_labels_arr = np.array(ctx_desc, dtype=ann.dtype)
brain_mask = binary_fill_holes(ann > 0)
ctx_mask = np.isin(ann, ctx_labels_arr)
core_mask = brain_mask & ~ctx_mask
print(f'core mask voxels: {core_mask.sum():,}')

# Gaussian-blur the binary mask so marching cubes produces a sub-voxel
# smooth iso-surface instead of stepped polygons.
core_float = gaussian_filter(core_mask.astype(np.float32), sigma=1.5)
verts, faces, _, _ = measure.marching_cubes(
    core_float, level=0.5, spacing=(RES_UM, RES_UM, RES_UM))
core = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
print(f'core raw: vertices={len(core.vertices):,}  '
      f'faces={len(core.faces):,}')
trimesh.smoothing.filter_taubin(core, iterations=20)

# Decimate via pyvista (fast_simplification not required).
try:
    pv_in = pv.wrap(core)
    pv_dec = pv_in.decimate(target_reduction=0.85)
    core = trimesh.Trimesh(vertices=np.asarray(pv_dec.points),
                            faces=pv_dec.regular_faces, process=True)
    print(f'decimated: {len(core.faces):,} faces')
except Exception as e:
    print(f'decimate failed: {e}')

# Label each core vertex by its MAJOR_DIVISION ancestor.
verts_um = core.vertices.copy()
vox = np.clip(np.round(verts_um / RES_UM).astype(int),
               0, np.array(ann.shape) - 1)
leaf_labels = ann[vox[:, 0], vox[:, 1], vox[:, 2]].astype(np.int64)
inward = verts_um - 200.0 * core.vertex_normals
vox2 = np.clip(np.round(inward / RES_UM).astype(int),
                0, np.array(ann.shape) - 1)
leaf2 = ann[vox2[:, 0], vox2[:, 1], vox2[:, 2]].astype(np.int64)
leaf_labels = np.where(leaf_labels == 0, leaf2, leaf_labels)

leaf_to_majdiv = {0: 0}
for L in np.unique(leaf_labels):
    if L == 0: continue
    try:
        path = tree.get_structures_by_id([int(L)])[0]['structure_id_path']
    except Exception:
        path = []
    match = [s for s in path if int(s) in MAJDIV_IDS]
    leaf_to_majdiv[int(L)] = int(match[0]) if match else 0
core_vert_div = np.array([leaf_to_majdiv[int(l)] for l in leaf_labels],
                          dtype=np.int64)
labeled = core_vert_div != 0
if (~labeled).any() and labeled.any():
    kd = cKDTree(verts_um[labeled])
    _, ii = kd.query(verts_um[~labeled])
    src = np.where(labeled)[0]
    core_vert_div[~labeled] = core_vert_div[src[ii]]

# Majority-voting label smoothing (regularizes boundary curves).
N = len(core.vertices)
edges = core.edges_unique
rows = np.concatenate([edges[:, 0], edges[:, 1]])
cols = np.concatenate([edges[:, 1], edges[:, 0]])
adj_c = csr_matrix((np.ones(len(rows), dtype=np.int32), (rows, cols)),
                    shape=(N, N))
for it in range(8):
    ua = np.array(sorted(set(core_vert_div.tolist())))
    a2c = {int(a): i for i, a in enumerate(ua)}
    own = np.array([a2c[int(L)] for L in core_vert_div])
    counts = np.zeros((N, len(ua)), dtype=np.int32)
    for j, A in enumerate(ua):
        counts[:, j] = adj_c @ (core_vert_div == int(A)).astype(np.int32)
    counts[np.arange(N), own] += 1
    core_vert_div = ua[np.argmax(counts, axis=1)]

print('divisions present on core surface:')
for sid in sorted(set(core_vert_div)):
    if sid == 0: continue
    acro, name = MAJOR_DIVISIONS[sid]
    n = int((core_vert_div == sid).sum())
    print(f'  {sid:4d}  {acro:<6s}  {name:<25s}  '
          f'{n:6,} verts ({100*n/len(core_vert_div):.1f}%)')

In [ ]:
# Option A: core with smooth-groove surface engraving (same trick as
# the cortex hat). Etch a bit deeper so the grooves are visible through
# the per-region colors.
core_print = core.copy()
core_print.apply_scale(SHARED_SCALE)
core_print.apply_translation(-root_ctr)

cfv = core_vert_div[core_print.faces]
core_bf = (cfv[:, 0] != cfv[:, 1]) | \
           (cfv[:, 0] != cfv[:, 2]) | \
           (cfv[:, 1] != cfv[:, 2])
core_bv_idx = np.unique(core_print.faces[core_bf].ravel())
print(f'core boundary faces: {core_bf.sum():,}  '
      f'boundary verts: {len(core_bv_idx):,}')

ETCH_DEPTH_CORE_MM   = 1.5
GROOVE_HW_CORE_MM    = 0.9
kd_bd = cKDTree(core_print.vertices[core_bv_idx])
dist, _ = kd_bd.query(core_print.vertices, k=1)
profile = np.cos(np.pi/2 * np.clip(dist / GROOVE_HW_CORE_MM, 0, 1)) ** 2
core_print.vertices = (core_print.vertices
                        - core_print.vertex_normals
                        * (ETCH_DEPTH_CORE_MM * profile)[:, None])

core_stl_a = OUT_DIR / f'allen_ccf_core_etched_{int(TARGET_MM)}mm.stl'
core_print.export(core_stl_a)
print(f'wrote {core_stl_a.name} '
      f'({core_stl_a.stat().st_size/1024/1024:.1f} MB)')

# Per-face dominant division for coloring.
core_face_div = cfv[:, 0]
m12 = cfv[:, 1] == cfv[:, 2]
m01 = cfv[:, 0] == cfv[:, 1]
core_face_div = np.where(m12 & ~m01, cfv[:, 1], core_face_div)

DIV_COLORS = {
    623:  '#5dd39e', 1089: '#ff9f1c', 703:  '#bdb2ff',
    698:  '#ffd166', 549:  '#cf6cf6', 1097: '#f08080',
    313:  '#5a9bff', 1065: '#118ab2', 512:  '#fa8072',
}
core_face_rgb = np.array([to_rgb(DIV_COLORS.get(int(d), '#888888'))
                           for d in core_face_div])

In [ ]:
# Option B: sagittal mid-line cut. Slice the *smoothed* core at the
# midline plane. Replace the coarse-cap rendering with a FINE
# rasterized "label slab" — one small colored quad per midline voxel
# — so the cut face shows brain regions at voxel resolution rather
# than a few huge polygons.

midline_z_um = ann.shape[2] / 2 * RES_UM
half = core.copy().slice_plane(plane_origin=[0, 0, midline_z_um],
                                plane_normal=[0, 0, -1], cap=True)
half_print = half.copy()
half_print.apply_scale(SHARED_SCALE)
half_print.apply_translation(-root_ctr)
half_stl_b = OUT_DIR / f'allen_ccf_core_sagittal_half_{int(TARGET_MM)}mm.stl'
half_print.export(half_stl_b)
print(f'wrote {half_stl_b.name} '
      f'({half_stl_b.stat().st_size/1024/1024:.1f} MB)')

# Build a "no-cap" outer half mesh (drop the coarse cap faces) for
# rendering — we'll overlay the fine slab in place of the cap.
half_tri_z = half.triangles[..., 2]
on_cut = np.abs(half_tri_z.mean(axis=1) - midline_z_um) < (RES_UM * 0.6)
keep_idx = np.where(~on_cut)[0]
half_outer = trimesh.Trimesh(vertices=half.vertices.copy(),
                              faces=half.faces[keep_idx].copy(),
                              process=False)
half_outer.apply_scale(SHARED_SCALE)
half_outer.apply_translation(-root_ctr)
half_outer_rgb = np.full((len(half_outer.faces), 3),
                           (0.91, 0.88, 0.83))

# Fine rasterized slab at z = midline_z. For each voxel of the
# annotation slice at midline that's INSIDE the brain, emit a quad
# (2 triangles) coloured by major division. Include cortex (688)
# too so the whole brain cross-section is colored.
slice_z_idx = int(round(midline_z_um / RES_UM))
slice_ann = ann[..., slice_z_idx]
DIV_COLORS_SLAB = dict(DIV_COLORS)
DIV_COLORS_SLAB[688] = '#d8b4fe'   # pale purple for isocortex
ctx_set = set(int(x) for x in ctx_desc)

unique_slice_leaves = np.unique(slice_ann)
slice_l2d = np.zeros(int(unique_slice_leaves.max()) + 1, dtype=np.int64)
for L in unique_slice_leaves:
    if L == 0: continue
    if int(L) in leaf_to_majdiv and leaf_to_majdiv[int(L)] != 0:
        slice_l2d[int(L)] = leaf_to_majdiv[int(L)]
    elif int(L) in ctx_set:
        slice_l2d[int(L)] = 688
    else:
        try:
            path = tree.get_structures_by_id([int(L)])[0]['structure_id_path']
        except Exception:
            path = []
        match = [s for s in path if int(s) in MAJDIV_IDS]
        slice_l2d[int(L)] = int(match[0]) if match else 0
slice_div = slice_l2d[slice_ann]
brain_slice = brain_mask[..., slice_z_idx]
slab_voxels = np.argwhere(brain_slice & (slice_div > 0))
print(f'slab voxels: {len(slab_voxels):,}')

H = RES_UM
verts_slab = []
faces_slab = []
slab_rgb = []
for (i, j) in slab_voxels:
    x0, x1 = i * H, (i + 1) * H
    y0, y1 = j * H, (j + 1) * H
    z = midline_z_um
    base = len(verts_slab)
    verts_slab.extend([(x0, y0, z), (x1, y0, z),
                        (x1, y1, z), (x0, y1, z)])
    faces_slab.append([base, base + 1, base + 2])
    faces_slab.append([base, base + 2, base + 3])
    col = to_rgb(DIV_COLORS_SLAB[int(slice_div[i, j])])
    slab_rgb.append(col); slab_rgb.append(col)
verts_slab = np.array(verts_slab, dtype=float)
faces_slab = np.array(faces_slab, dtype=np.int64)
slab_rgb = np.array(slab_rgb)
slab = trimesh.Trimesh(vertices=verts_slab, faces=faces_slab,
                        process=False)
slab.apply_scale(SHARED_SCALE)
slab.apply_translation(-root_ctr)
slab.vertices[:, 2] -= 0.05   # tiny offset so it doesn't z-fight the cap
print(f'slab: {len(slab.faces):,} colored triangles')

# half_face_rgb is left around for backwards compat (not used now).
half_face_rgb = None

In [ ]:
# Render both options + legend.
def make_poly(mesh, face_rgb):
    poly = pv.PolyData(
        mesh.vertices,
        np.hstack([np.full((len(mesh.faces), 1), 3),
                     mesh.faces]).astype(np.int64))
    poly.cell_data['rgb'] = (face_rgb * 255).astype(np.uint8)
    return poly

bb_core = core_print.bounding_box.extents
d = bb_core.max() * 2.2

# Option A: 2x4 grid: top row colored by region, bottom row solid so
# the groove geometry pops.
core_face_rgb_solid = np.full_like(core_face_rgb, fill_value=0.85)
views_a = [
    ('dorsal',  [(0, -d, 0), (0, 0, 0), (-1, 0,  0)]),
    ('right',   [(0,  0,  d), (0, 0, 0), ( 0, -1, 0)]),
    ('caudal',  [(d,  0,  0), (0, 0, 0), ( 0, -1, 0)]),
    ('oblique', [(0.7*d, -0.5*d, 0.7*d), (0, 0, 0), (0, -1, 0)]),
]
pl = pv.Plotter(shape=(2, 4), window_size=(2200, 1200),
                 off_screen=True)
for k, (vname, cam) in enumerate(views_a):
    pl.subplot(0, k)
    pl.add_mesh(make_poly(core_print, core_face_rgb),
                 scalars='rgb', rgb=True, smooth_shading=True,
                 specular=0.20, ambient=0.22, diffuse=0.95)
    pl.add_text(f'colored — {vname}', font_size=9)
    pl.camera_position = cam; pl.reset_camera()
    pl.subplot(1, k)
    pl.add_mesh(make_poly(core_print, core_face_rgb_solid),
                 scalars='rgb', rgb=True, smooth_shading=True,
                 specular=0.20, ambient=0.20, diffuse=0.95)
    pl.add_text(f'solid — {vname}', font_size=9)
    pl.camera_position = cam; pl.reset_camera()
pl.background_color = 'white'
pl.screenshot(str(OUT_DIR / 'core_optA_surface_engraved.png'))
pl.close()

# Option B: 1x4 grid showing the cut face (with fine colored slab)
# from multiple angles + the outer side + dorsal.
# Camera: cut face at z = midline_z*SHARED_SCALE - root_ctr[2] ≈ 0;
# kept half has z < 0; so to *see* the cut face, camera is at z > 0.
views_b = [
    ('cut face',     [(0, 0,  d), (0, 0, 0), (0, -1, 0)]),
    ('oblique cut',  [(0.3*d, -0.3*d,  0.9*d), (0, 0, 0), (0, -1, 0)]),
    ('outer (left)', [(0, 0, -d), (0, 0, 0), (0, -1, 0)]),
    ('dorsal',       [(0, -d, 0), (0, 0, 0), (-1, 0, 0)]),
]
pl = pv.Plotter(shape=(1, 4), window_size=(2200, 700), off_screen=True)
for k, (vname, cam) in enumerate(views_b):
    pl.subplot(0, k)
    pl.add_mesh(make_poly(half_outer, half_outer_rgb),
                 scalars='rgb', rgb=True, smooth_shading=True,
                 specular=0.15, ambient=0.30, diffuse=0.85)
    pl.add_mesh(make_poly(slab, slab_rgb),
                 scalars='rgb', rgb=True, smooth_shading=False,
                 specular=0.05, ambient=0.55, diffuse=0.55)
    pl.add_text(vname, font_size=10)
    pl.camera_position = cam; pl.reset_camera()
pl.background_color = 'white'
pl.screenshot(str(OUT_DIR / 'core_optB_sagittal_half.png'))
pl.close()

# Legend (now includes CTX for Option B's slab).
legend_fig = plt.figure(figsize=(8, 2.2), dpi=140, facecolor='white')
ax = legend_fig.add_subplot(111); ax.axis('off')
order = [688, 623, 1089, 703, 698, 549, 1097, 313, 1065, 512]
LEG_NAMES = {688: ('CTX', 'Isocortex'), **MAJOR_DIVISIONS}
LEG_COLORS = DIV_COLORS_SLAB
for i, sid in enumerate(order):
    acro, name = LEG_NAMES[sid]
    color = LEG_COLORS[sid]
    x = (i % 5) * 0.20
    y = 1.0 - (i // 5) * 0.45
    ax.add_patch(plt.Rectangle((x, y - 0.18), 0.04, 0.18,
                                 facecolor=color, edgecolor='black',
                                 lw=0.5, transform=ax.transAxes))
    ax.text(x + 0.05, y - 0.09, f'{acro} — {name}',
             transform=ax.transAxes, va='center', fontsize=9)
legend_fig.suptitle('Brain divisions', fontsize=11)
legend_fig.savefig(OUT_DIR / 'core_division_legend.png',
                    dpi=140, bbox_inches='tight', facecolor='white')
plt.close(legend_fig)

from IPython.display import Image, display
print('\n=== Option A: core with surface engraving ===')
display(Image(filename=str(OUT_DIR / 'core_optA_surface_engraved.png')))
print('\n=== Option B: sagittal mid-line cut (colored cut face) ===')
display(Image(filename=str(OUT_DIR / 'core_optB_sagittal_half.png')))
print('\n=== legend ===')
display(Image(filename=str(OUT_DIR / 'core_division_legend.png')))